# Train YOLOv8 (feces-only) on Colab GPU

Trains the single-class **feces** YOLOv8 detector on a free Google Colab/T4 GPU,
evaluates it against the **mAP50 >= 0.80** target, and copies a downloadable
`training-outputs.zip` back to your mounted Google Drive.

### Run order
1. **Runtime -> Change runtime type -> GPU (T4)**.
2. Run all cells in order.
3. When finished, download `training-outputs.zip` from Drive (path printed at the end)
   and unzip it into the repo `training/` folder.


In [ ]:
# 1) Mount Google Drive (open the link, approve, paste the code)
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")


In [ ]:
# 2) Configuration
import os

# Dataset zip you uploaded to Drive (edit if in a subfolder).
DRIVE_ZIP = "/content/drive/MyDrive/pigfeces-yolo.zip"
assert os.path.exists(DRIVE_ZIP), "Upload pigfeces-yolo.zip to Drive first."

# Where the final outputs zip is written back to Drive.
DRIVE_OUT = "/content/drive/MyDrive/training-outputs.zip"

# Repo URL to clone the training scripts (yields train.py/evaluate.py/dataset.yaml).
# The dataset itself is git-ignored, so it is NOT in git - the zip above provides it.
REPO_URL = "https://github.com/renumbracode/feces-detector-ai"
assert REPO_URL, "Set REPO_URL to your repo (any public/fork)."

print("OK")


In [ ]:
# 3) Clone the repo to fetch training/ scripts + dataset.yaml
import os
repo = "/content/feces-detector-ai"
if os.path.exists(repo) and os.listdir(repo):
    print("repo already present:", repo)
else:
    !git clone {REPO_URL} {repo}
assert os.path.exists(f"{repo}/training/train.py"), "train.py not found after clone"
print("Scripts ready.")


In [ ]:
# 4) Unzip the dataset into $REPO/dataset
import os, zipfile
repo = "/content/feces-detector-ai"
os.makedirs(f"{repo}/dataset", exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP) as z:
    z.extractall(f"{repo}/dataset")
base = f"{repo}/dataset"
for d in ["images/train","images/val","labels/train","labels/val"]:
    p = f"{base}/{d}"
    print(f"{d}:", len(os.listdir(p)) if os.path.isdir(p) else "MISSING")


In [ ]:
# 5) Point dataset.yaml path: at the Colab dataset location
import os, re, pathlib
repo = "/content/feces-detector-ai"
DATA_YAML = f"{repo}/training/dataset.yaml"
text = pathlib.Path(DATA_YAML).read_text()
abs_path = f"{repo}/dataset".replace("\\", "/")
text = re.sub(r"path:\s*\S+", f"path: {abs_path}", text)
pathlib.Path(DATA_YAML).write_text(text)
print(text)


In [ ]:
# 6) Install dependencies (ultralytics pulls torch + CUDA)
!pip -q install ultralytics
import torch, ultralytics
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Train

100 epochs @ 640, batch 16. On a free Colab T4 expect **~40-90 minutes**
(early-stopping patience 30 may finish sooner). Leave this cell running until it completes.


In [ ]:
# 7) TRAIN (long step)
repo = "/content/feces-detector-ai"
!python {repo}/training/train.py \
      --model yolov8n.pt \
      --epochs 100 \
      --imgsz 640 \
      --batch 16 \
      --patience 30 \
      --project {repo}/training/runs \
      --name detect \
      --device 0 \
      --exist-ok
print("Training finished.")


In [ ]:
# 8) EVALUATE against the 0.80 target
repo = "/content/feces-detector-ai"
!python {repo}/training/evaluate.py \
      --model {repo}/training/runs/detect/weights/best.pt \
      --data {repo}/training/dataset.yaml \
      --target 0.80 \
      --device 0
print("Evaluate done. See evaluation_summary.txt above.")


In [ ]:
# 9) Package outputs and copy back to Drive
import os, zipfile, shutil
repo = "/content/feces-detector-ai"
det = f"{repo}/training/runs/detect"
print("run contents:", os.listdir(det))

zpath = "/content/training-outputs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    zw = f"{det}/weights"
    for name in ("best.pt", "last.pt"):
        p = f"{zw}/{name}"
        if os.path.exists(p):
            z.write(p, f"weights/{name}")
    # metrics
    for name in ("results.csv", "evaluation_summary.txt"):
        p = f"{det}/{name}"
        if os.path.exists(p):
            z.write(p, name)
    # a couple diagnostic plots if present
    for name in os.listdir(det):
        if name in ("labels.jpg", "results.png", "confusion_matrix.png", "PR_curve.png"):
            z.write(f"{det}/{name}", name)

shutil.copy(zpath, DRIVE_OUT)
print("DOWNLOAD READY ->", DRIVE_OUT)


## After training

Unzip `training-outputs.zip` into the repo so that
`training/runs/detect/weights/best.pt` matches the verify-server default (`YOLO_MODEL`),
then start the verify server and the dashboard.
